<p style="font-size:24px;font-family:Arial;color:#FFA500"><b>Introduction</b></p>

<p style="font-size:16px;font-family:Arial;color:#FFA500">
This notebook provides authentication methods for accessing Teradata AI services. It supports two primary authentication approaches:
</p>

<ol style="font-size:16px;font-family:Arial;color:#FFA500">
<li><b>Device Flow Authentication (REQUIRED for VectorStore)</b> - Browser-based JWT token generation using OAuth 2.0 Device Flow</li>
<li><b>Username/Password Authentication</b> - Direct authentication using Keycloak or Ping IDP credentials (for general database access)</li>
</ol>

<div class="alert alert-block alert-warning">
<p style='font-size:14px;font-family:Arial;color:#FFA500'><b>⚠️ Important:</b> <b>VectorStore operations require JWT token authentication</b> generated via Device Flow. Username/password authentication alone is not supported for VectorStore APIs.</p>
</div>

<p style="font-size:16px;font-family:Arial;color:#FFA500">
<b>Authentication Flow Summary:</b>
</p>
<ul style="font-size:14px;font-family:Arial;color:#FFA500">
<li><b>For VectorStore:</b> Use Device Flow to generate JWT token → Store in <code>user_token</code> variable → Pass to VectorStore operations</li>
<li><b>For Database (create_context):</b> Can use either JWT token or username/password</li>
<li><b>Dual Auth Support:</b> This notebook generates a <code>user_token</code> variable that works with both authentication methods</li>
</ul>

<div class="alert alert-block alert-info">
<p style='font-size:14px;font-family:Arial;color:#FFA500'><b>ℹ️ Note:</b> This notebook sets up authentication tokens that can be used across other Teradata AI notebooks for VectorStore, TextAnalytics, and ModelOps operations.</p>
</div>

## 📋 Prerequisites

<p style='font-size:16px;font-family:Arial;color:#FFA500'>
Before proceeding, ensure you have:
</p>

<ul style='font-size:16px;font-family:Arial;color:#FFA500'>
<li>Access to Teradata AI platform (Teradata Vantage with AI/ML capabilities)</li>
<li>Valid user credentials (username and password)</li>
<li>The following service URLs from your Teradata administrator:
  <ul>
    <li>VectorStore Base URL</li>
    <li>Keycloak/IDP Base URL</li>
    <li>ModelOps Base URL</li>
  </ul>
</li>
</ul>

## ⚙️ Environment Setup

<p style='font-size:16px;font-family:Arial;color:#FFA500'>
First, we'll import required libraries and configure the authentication settings.
</p>

<div class="alert alert-block alert-info">
<p style='font-size:14px;font-family:Arial;color:#FFA500'><b>ℹ️ Configuration Note:</b> This notebook requires you to provide your Teradata service URLs. You can either enter them manually when prompted, or set environment variables if you prefer automation.</p>
</div>

In [3]:
import os
import getpass
import requests
import json
import warnings
import time
import webbrowser
from IPython.display import display, Markdown

warnings.simplefilter('ignore')

### Configuration Setup

<p style='font-size:14px;font-family:Arial;color:#FFA500'>
The next cell contains two configuration methods. By default, it uses <b>auto_configure_from_env()</b> which reads from environment variables.
</p>

<p style='font-size:14px;font-family:Arial;color:#FFA500'>
<b>To switch to manual input:</b> Comment out the auto_configure line and uncomment the manual_configure line.
</p>

<hr style="height:2px;border:none;background-color:#FFA500;">

In [ ]:
# ====================================================================
# CONFIGURATION FUNCTIONS - Define before using
# ====================================================================

# Option 1: Auto-configure from environment variables (JupyterHub / AI Workbench)
def auto_configure_from_env():
    """
    Load configuration from environment variables (NO TD_ prefix).
    TD_ prefix is ONLY for database connection parameters.
    """
    config = {}
    
    # Configure SSL verification
    ssl_verify_env = os.getenv('SSL_VERIFY', "true")
    config['SSL_VERIFY'] = ssl_verify_env.lower() in ["true", "1", "yes"]
    
    # Configure base URLs
    vs_url = os.getenv('VECTORSTORE_BASE_URL', '')
    vs_url = vs_url.rstrip('/')
    if vs_url.endswith('/data-insights'):
        vs_url = vs_url[:-len('/data-insights')]
    config['VS_BASE_URL'] = vs_url
    
    config['KEYCLOAK_BASE_URL'] = os.getenv('KEYCLOAK_BASE_URL', '').rstrip('/')
    config['MODELOPS_BASE_URL'] = os.getenv('MODELOPS_BASE_URL', '')
    
    # IDP Configuration
    config['IDP_TYPE'] = os.getenv('IDP_TYPE', 'keycloak')
    config['IDP_CLIENT_ID'] = os.getenv('IDP_CLIENT_ID', 'vectorstore')
    config['IDP_BASE_URL'] = os.getenv('IDP_BASE_URL', config['KEYCLOAK_BASE_URL'])
    
    # Site identification
    jh_fqdn = os.getenv('JH_FQDN', '')
    config['SITE_ID'] = jh_fqdn.split('.')[0].upper() if jh_fqdn else ''
    
    config['REALM'] = os.getenv('REALM', 'teradata')
    
    return config

# Option 2: Manual configuration via user input
def manual_configure():
    """Prompt user for configuration values"""
    config = {}
    
    print("🔧 Manual Configuration Setup")
    print("=" * 60)
    print("\nPlease provide the following configuration details:\n")
    
    # VectorStore Base URL
    config['VS_BASE_URL'] = input("Enter VectorStore Base URL (e.g., https://your-site.teradata.com): ").strip().rstrip('/')
    if config['VS_BASE_URL'].endswith('/data-insights'):
        config['VS_BASE_URL'] = config['VS_BASE_URL'][:-len('/data-insights')]
    
    # Keycloak/IDP Base URL
    config['KEYCLOAK_BASE_URL'] = input("Enter Keycloak/IDP Base URL (e.g., https://keycloak.your-site.com): ").strip().rstrip('/')
    
    # ModelOps Base URL
    config['MODELOPS_BASE_URL'] = input("Enter ModelOps Base URL (e.g., https://modelops.your-site.com): ").strip().rstrip('/')
    
    # IDP Type
    print("\nSelect IDP Type:")
    print("  1. Keycloak (default)")
    print("  2. PingFederate")
    idp_choice = input("Enter choice (1 or 2) [default: 1]: ").strip() or "1"
    config['IDP_TYPE'] = 'keycloak' if idp_choice == '1' else 'ping'
    
    # Site ID (for PingFederate)
    if config['IDP_TYPE'] == 'ping':
        config['SITE_ID'] = input("Enter Site ID (for PingFederate): ").strip().upper()
        config['IDP_CLIENT_ID'] = f"onetd-dev-{config['SITE_ID']}"
        config['IDP_BASE_URL'] = input("Enter PingFederate Base URL: ").strip().rstrip('/')
    else:
        config['IDP_CLIENT_ID'] = 'vectorstore'
        config['IDP_BASE_URL'] = config['KEYCLOAK_BASE_URL']
        config['SITE_ID'] = ''
    
    # SSL Verification
    print("\nEnable SSL Certificate Verification?")
    print("  1. Yes (recommended for production)")
    print("  2. No (only for development/testing)")
    ssl_choice = input("Enter choice (1 or 2) [default: 1]: ").strip() or "1"
    config['SSL_VERIFY'] = ssl_choice == '1'
    
    config['REALM'] = "teradata"
    
    print("\n" + "=" * 60)
    print("✅ Configuration completed!\n")
    
    return config

# ====================================================================
# EXECUTE CONFIGURATION - Choose one option below
# ====================================================================

# CHOOSE ONE OPTION:

# Option 1: Environment Variables (Recommended for JupyterHub)
config = auto_configure_from_env()

# Option 2: Manual Input (Uncomment to use)
# config = manual_configure()

# ====================================================================
# Extract and verify configuration values
# ====================================================================

SSL_VERIFY = config['SSL_VERIFY']
VS_BASE_URL = config['VS_BASE_URL']
KEYCLOAK_BASE_URL = config['KEYCLOAK_BASE_URL']
MODELOPS_BASE_URL = config['MODELOPS_BASE_URL']
IDP_TYPE = config['IDP_TYPE']
IDP_CLIENT_ID = config['IDP_CLIENT_ID']
SITE_ID = config.get('SITE_ID', '')
IDP_BASE_URL = config.get('IDP_BASE_URL', '')
REALM = config['REALM']

# Display configuration summary
print("\n" + "=" * 60)
print("📋 Configuration Summary:")
print("=" * 60)
print(f"✅ VectorStore Base URL: {VS_BASE_URL if VS_BASE_URL else '[NOT SET]'}")
print(f"✅ Keycloak Base URL: {KEYCLOAK_BASE_URL if KEYCLOAK_BASE_URL else '[NOT SET]'}")
print(f"✅ ModelOps Base URL: {MODELOPS_BASE_URL if MODELOPS_BASE_URL else '[NOT SET]'}")
print(f"✅ IDP Type: {IDP_TYPE}")
print(f"✅ IDP Client ID: {IDP_CLIENT_ID}")
print(f"✅ IDP Base URL: {IDP_BASE_URL if IDP_BASE_URL else '[SAME AS KEYCLOAK]'}")
print(f"✅ Realm: {REALM}")
print(f"✅ Site ID: {SITE_ID if SITE_ID else '[AUTO-DETECTED]'}")
print(f"✅ SSL Verification: {SSL_VERIFY}")
print("=" * 60)
print("\n✅ Configuration complete! You can now proceed to authentication.\n")


📋 Configuration Summary:
✅ VectorStore Base URL: https://aiop-blueberry-tmsb.td.teradata.com
✅ Keycloak Base URL: https://aiop-blueberry-tmsb.td.teradata.com/sso
✅ ModelOps Base URL: https://aiop-blueberry-tmsb.td.teradata.com
✅ SSL Verification: True
✅ IDP Type: keycloak
✅ Client ID: vectorstore

✅ Configuration complete! You can now proceed to authentication.



## 🔐 Authentication Methods

<p style='font-size:16px;font-family:Arial;color:#FFA500'>
Select one of the following authentication methods based on your requirements:
</p>

### Method 1: Username/Password Authentication

<p style='font-size:16px;font-family:Arial;color:#FFA500'>
This method allows direct authentication using your IDP credentials (Keycloak or PingFederate).
</p>

<p style='font-size:14px;font-family:Arial;color:#FFA500'>
<b>Use this method when:</b>
</p>
<ul style='font-size:14px;font-family:Arial;color:#FFA500'>
<li>You want quick, interactive authentication</li>
<li>You're working in a trusted environment</li>
<li>You don't require additional security layers</li>
</ul>

In [ ]:
def gen_user_token():
    """
    Generate authentication token using username and password.
    
    Returns:
        str: JWT access token
    """
    if IDP_TYPE == "ping":
        KEYCLOAK_USER = input("Enter IDP username: ")
        KEYCLOAK_PASSWORD = getpass.getpass("Enter IDP user's password: ")
        url = f"{IDP_BASE_URL}/as/token.oauth2"
    else:
        KEYCLOAK_USER = "<keycloak_user>"
        KEYCLOAK_PASSWORD = "<keycloak_password>"
        url = f"{KEYCLOAK_BASE_URL}/realms/{REALM}/protocol/openid-connect/token"

    payload = {
        "grant_type": "password",
        "client_id": IDP_CLIENT_ID,
        "username": KEYCLOAK_USER,
        "password": KEYCLOAK_PASSWORD
    }
    
    try:
        response = requests.post(url, data=payload, verify=SSL_VERIFY)
        response.raise_for_status()
        access_token = response.json()["access_token"]
        print("✅ Authentication successful! Token generated.")
        return access_token
    except requests.exceptions.HTTPError as e:
        print(f"❌ Authentication failed: {e}")
        raise
    except Exception as e:
        print(f"❌ Error during authentication: {e}")
        raise

# Uncomment the line below to use this authentication method
# user_token = gen_user_token()
# print(f"Token: {user_token[:50]}...")

### Method 2: Device Flow Authentication (REQUIRED for VectorStore)

<p style='font-size:16px;font-family:Arial;color:#FFA500'>
This method uses OAuth 2.0 Device Authorization Flow to generate a JWT token, which is <b>required for VectorStore operations</b>.
</p>

<div class="alert alert-block alert-warning">
<p style='font-size:14px;font-family:Arial;color:#FFA500'><b>⚠️ Required for VectorStore:</b> VectorStore APIs do not support username/password authentication. You <b>must</b> use Device Flow to generate a JWT token.</p>
</div>

<p style='font-size:14px;font-family:Arial;color:#FFA500'>
<b>Use this method when:</b>
</p>
<ul style='font-size:14px;font-family:Arial;color:#FFA500'>
<li><b>Working with VectorStore (REQUIRED)</b></li>
<li>You need enhanced security with multi-factor authentication (MFA)</li>
<li>You want to avoid entering passwords directly in the notebook</li>
<li>You're working in JupyterHub environment</li>
<li>Your organization requires browser-based authentication</li>
</ul>

<p style='font-size:14px;font-family:Arial;color:#FFA500'>
<b>How it works:</b>
</p>
<ol style='font-size:14px;font-family:Arial;color:#FFA500'>
<li>The script generates a unique URL and device code</li>
<li>A browser window opens automatically (or you can click the link)</li>
<li>You authenticate through your IDP (Keycloak/Ping) in the browser</li>
<li>After successful authentication, the JWT token is automatically retrieved and stored in <code>user_token</code> variable</li>
<li>This token can then be used with <code>set_auth_token(user_token)</code> for VectorStore operations</li>
</ol>

In [9]:
def gen_user_token_using_device_flow():
    """
    Generate authentication token using OAuth 2.0 Device Flow.
    Opens a browser for authentication and polls for token.
    
    Returns:
        str: JWT access token
    """
    # Step 1: Request device + user code
    if IDP_TYPE == "ping":
        device_auth_url = f"{IDP_BASE_URL}/as/device_authz.oauth2"
        token_url = f"{IDP_BASE_URL}/as/token.oauth2"
    else:
        device_auth_url = f"{KEYCLOAK_BASE_URL}/realms/{REALM}/protocol/openid-connect/auth/device"
        token_url = f"{KEYCLOAK_BASE_URL}/realms/{REALM}/protocol/openid-connect/token"

    try:
        response = requests.post(device_auth_url, data={
            "client_id": IDP_CLIENT_ID,
            "scope": "openid"
        }, verify=SSL_VERIFY)
        
        response.raise_for_status()
        data = response.json()
        
        # Show user-friendly prompt in notebook
        display(Markdown(f"""
### 🔐 Authenticate Using Device Flow

Please authenticate to get your access token:

**[🌐 Click here to authenticate]({data['verification_uri_complete']})**

Or open `{data['verification_uri']}` and enter this code:

### `{data['user_code']}`

*Waiting for authentication...*
        """))
        
        # Optionally open browser tab automatically
        try:
            webbrowser.open(data['verification_uri_complete'])
        except:
            pass  # Silently fail if browser can't be opened
        
        device_code = data["device_code"]
        interval = data.get("interval", 5)
        
        # Step 2: Poll token endpoint
        while True:
            time.sleep(interval)
            token_response = requests.post(token_url, data={
                "grant_type": "urn:ietf:params:oauth:grant-type:device_code",
                "device_code": device_code,
                "client_id": IDP_CLIENT_ID
            }, verify=SSL_VERIFY)
            
            if token_response.status_code == 200:
                tokens = token_response.json()
                access_token = tokens['access_token']
                display(Markdown(f"""
### ✅ Authentication Successful!

Your access token has been generated and is ready to use.

**Token Preview:** `{access_token[:50]}...`
                """))
                return access_token
            else:
                error = token_response.json().get("error")
                if error == "authorization_pending":
                    print("⏳ Waiting for user authorization...")
                    continue
                elif error == "slow_down":
                    interval += 5
                    continue
                else:
                    display(Markdown(f"❌ **Authentication Error:** `{error}`"))
                    raise Exception(f"Authentication failed: {error}")
    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP Error during authentication: {e}")
        raise
    except Exception as e:
        print(f"❌ Error during device flow authentication: {e}")
        raise

# Uncomment the line below to use this authentication method
# user_token = gen_user_token_using_device_flow()

## 🚀 Execute Authentication

<p style='font-size:16px;font-family:Arial;color:#FFA500'>
Run the cell below to authenticate using your preferred method (Device Flow is recommended).
</p>

In [ ]:
# Choose your authentication method:
# Option 1: Device Flow (REQUIRED for VectorStore)
user_token = gen_user_token_using_device_flow()

# Option 2: Username/Password (NOT supported for VectorStore)
#user_token = gen_user_token()

print("\n✅ Authentication complete! Your token is stored in the 'user_token' variable.")
print("You can now use this token in other notebooks or API calls.")


### 🔐 Authenticate Using Device Flow

Please authenticate to get your access token:

**[🌐 Click here to authenticate](https://aiop-blueberry-tmsb.td.teradata.com/sso/realms/teradata/device?user_code=RUXH-QFNP)**

Or open `https://aiop-blueberry-tmsb.td.teradata.com/sso/realms/teradata/device` and enter this code:

### `RUXH-QFNP`

*Waiting for authentication...*
        


### ✅ Authentication Successful!

Your access token has been generated and is ready to use.

**Token Preview:** `eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6IC...`
                


✅ Authentication complete! Your token is stored in the 'user_token' variable.
You can now use this token in other notebooks or API calls.


## 🔍 Verify Authentication

<p style='font-size:16px;font-family:Arial;color:#FFA500'>
You can verify that your authentication token is valid and view some basic information about it.
</p>

In [11]:
import base64

def decode_jwt_payload(token):
    """
    Decode and display JWT token payload information.
    
    Args:
        token (str): JWT token
    """
    try:
        # JWT tokens have three parts separated by dots
        parts = token.split('.')
        if len(parts) != 3:
            print("❌ Invalid token format")
            return
        
        # Decode the payload (second part)
        # Add padding if necessary
        payload = parts[1]
        padding = 4 - len(payload) % 4
        if padding != 4:
            payload += '=' * padding
        
        decoded = base64.urlsafe_b64decode(payload)
        payload_data = json.loads(decoded)
        
        print("✅ Token Information:")
        print(f"   - Subject (User): {payload_data.get('sub', 'N/A')}")
        print(f"   - Preferred Username: {payload_data.get('preferred_username', 'N/A')}")
        print(f"   - Email: {payload_data.get('email', 'N/A')}")
        
        if 'exp' in payload_data:
            exp_time = payload_data['exp']
            from datetime import datetime
            exp_datetime = datetime.fromtimestamp(exp_time)
            print(f"   - Token Expires: {exp_datetime}")
        
        if 'iat' in payload_data:
            iat_time = payload_data['iat']
            iat_datetime = datetime.fromtimestamp(iat_time)
            print(f"   - Token Issued: {iat_datetime}")
            
    except Exception as e:
        print(f"⚠️ Could not decode token: {e}")

# Verify the token
if 'user_token' in locals():
    decode_jwt_payload(user_token)
else:
    print("⚠️ No token found. Please run the authentication cell first.")

✅ Token Information:
   - Subject (User): af9de0c7-7a5b-446e-8e2f-fe093e9d3ef6
   - Preferred Username: admin
   - Email: admin.modelops@aiop-blueberry-tmsb.td.teradata.com
   - Token Expires: 2025-12-18 02:21:25
   - Token Issued: 2025-12-17 16:45:14


## 📦 Using the Token in Your Applications

<p style='font-size:16px;font-family:Arial;color:#FFA500'>
The authentication token is stored in the <code>user_token</code> variable. You can use this token in your applications:
</p>

<ol style='font-size:16px;font-family:Arial;color:#FFA500'>
<li><b>Run this notebook using %run magic:</b> <code>%run Authentication_Guide.ipynb</code></li>
<li><b>Access the token variable:</b> Use <code>user_token</code> directly in subsequent cells</li>
<li><b>Pass the token to API clients:</b> See examples below</li>
</ol>

<div class="alert alert-block alert-info">
<p style='font-size:14px;font-family:Arial;color:#FFA500'><b>ℹ️ Dual Authentication Support:</b> The token-based authentication works alongside traditional username/password methods. Your applications can support both authentication mechanisms for flexibility.</p>
</div>


In [ ]:
# The user_token variable is already available for use in your scripts
# No need to set environment variables - just use the token directly

if 'user_token' in locals():
    print("✅ Token is available in the 'user_token' variable")
    print(f"   Token preview: {user_token[:50]}...")
    print("\n💡 Usage in VectorStore notebooks:")
    print("   1. Run: %run Authentication_Guide.ipynb")
    print("   2. Access: user_token variable will be available")
    print("\n📝 Example - Create database context with JWT:")
    print("   from teradataml import create_context")
    print("   host = VS_BASE_URL.replace('https://', '').split('/')[0]")
    print("   context = create_context(")
    print("       host=host,")
    print("       logmech='JWT',")
    print("       logdata=f'token={user_token}',")
    print("       base_url=VS_BASE_URL,")
    print("       auth_token=user_token")
    print("   )")
else:
    print("⚠️ No token found. Please run the authentication cell first.")


✅ Token is available in the 'user_token' variable
   Token preview: eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6IC...

💡 Usage in other notebooks:
   1. Run: %run Authentication_Guide.ipynb
   2. Access: user_token variable will be available
   3. Use directly in API calls (see examples below)


### 🔐 Dual Authentication Support

<p style='font-size:14px;font-family:Arial;color:#FFA500'>
This authentication pattern supports both JWT token-based and basic authentication methods:
</p>

<p style='font-size:14px;font-family:Arial;color:#FFA500'>
<b>Token-Based Authentication (Recommended):</b>
</p>
<ul style='font-size:13px;font-family:Arial;color:#FFA500'>
<li>Use the <code>user_token</code> variable generated from this notebook</li>
<li>Pass as Bearer token: <code>Authorization: Bearer {user_token}</code></li>
<li>Supports MFA and enhanced security</li>
<li>Works with all Teradata AI services (VectorStore, TextAnalytics, ModelOps)</li>
</ul>

<p style='font-size:14px;font-family:Arial;color:#FFA500'>
<b>Basic Authentication (Fallback):</b>
</p>
<ul style='font-size:13px;font-family:Arial;color:#FFA500'>
<li>Direct username/password credentials</li>
<li>Useful for legacy integrations or when token flow is not available</li>
<li>Can be used interchangeably with token-based auth</li>
</ul>


## 🔄 Token Refresh

<p style='font-size:16px;font-family:Arial;color:#FFA500'>
JWT tokens typically expire after a certain period (usually 1-24 hours). If your token expires, simply re-run the authentication cell to get a new token.
</p>

<div class="alert alert-block alert-warning">
<p style='font-size:14px;font-family:Arial;color:#FFA500'><b>⚠️ Security Note:</b> Never share your authentication token or commit it to version control. Tokens provide full access to your account.</p>
</div>

## 📖 Additional Resources

<p style='font-size:16px;font-family:Arial;color:#FFA500'>
For more information about Teradata AI services:
</p>

<ul style='font-size:16px;font-family:Arial;color:#FFA500'>
<li><a href="https://docs.teradata.com/">Teradata Documentation</a></li>
<li><a href="https://docs.teradata.com/r/Teradata-VantageCloud-Lake/AI-Unlimited">Teradata AI Unlimited</a></li>
<li><a href="https://docs.teradata.com/r/Enterprise_IntelliFlex_VMware/SQL-Functions/Teradata-Vantage-AI-Functions">Teradata AI Functions</a></li>
</ul>

---

<p style='font-size:12px;font-family:Arial;color:#666666;text-align:center'>
© 2024 Teradata Corporation. All Rights Reserved.
</p>